In [39]:
import os 
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY") #get API key from env file

BASE_URL = "https://www.alphavantage.co/query" #base url for api

def fetch_stock_data(symbol):
    
    """
    Fetch daily stock data from Alpha API
    """

    
    params = {
        "function" : "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey" : API_KEY 
    }

    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200 :
        raise Exception("API request dailed")

   
    data = response.json() 
    
    if "Error Message" in data:
        raise Exception("Invalid API call")

    if "Note" in data:
        raise Exception("API rate limit exceeded")

    return data 

def parse_stock_data(data) :
    """
    Extract time serise data from API response
    """

    if "Time Series (Daily)" not in data:
        raise Exception("Time Series data not found in API response")
    
    return data["Time Series (Daily)"]

import pandas as pd

def convert_to_df(df) :
    return pd.DataFrame(df).T


def time_serise_analysis_dataset (df) :
    df = df.copy() #copy this dataset so that for delink the referance 
    df = df.drop_duplicates() #drop the duplicate value
    df = df.dropna()  #drop the missing value KAN-39
    df = df.rename(columns={                            
            "1. open" :"open",
            "2. high" :"high" ,  
            "3. low"  :"low" , 
            "4. close" :"close"  ,
            "5. volume" : "volume" }) #rename the column
    df = df.astype({"open":float,"high":float,"low":float,"close":float,"volume":int}).round(2) #reasign the data type and round it by 2

    df.index = pd.to_datetime(df.index) #set index value as date time
    
    df = df.sort_index(ascending=False) # sorted the index

    df.reset_index(inplace=True)
    df.rename(columns={"index":"Date"},inplace=True) #change the index name as date 
    df = df.set_index("Date") #set this cloumn as date time 
    return df
def time_serise_feature_engineering(df) :
    
    df["close_norm"] = (df["close"] /  df["close"].max()).astype(float).round(2)  #normalized this stock value KAN-40 


    df["retuens"] = (df["close"].pct_change().fillna(0)).astype(float).round(4) #calculatethe daily return KAN-41

        

    df["pct_change"] = (df["close"].pct_change().fillna(0) * 100).astype(float).round(2) #calculate the percentage chnage
    
    return df


def compute_all_statistics(df) :
    stats = {}

    # core return stats
    stats["mean"] = round(float(df["retuens"].mean()),6)
    stats["median"] = round(float(df["retuens"].median()),6)
    stats["std_dev"] = round(float(df["retuens"].std()),6)
    stats["variance"] =  round(float(df["retuens"].var()),6)

    # Extremes
    stats["min"] =  round(float(df["retuens"].min()),6)
    stats["max"] =  round(float(df["retuens"].max()),6)

    #Percentiles
    stats["percentiles"] = {
        "25" :round(float(df["retuens"].quantile(0.25)),6),
        "50" :round(float(df["retuens"].quantile(0.50)),6),
        "75" :round(float(df["retuens"].quantile(0.75)),6),
        "5"  :round(float(df["retuens"].quantile(0.05)),6),
        "95" :round(float(df["retuens"].quantile(0.95)),6),

    }

    return stats

In [40]:
def get_stock_analysis(symbol) :

     data = fetch_stock_data(symbol)  #get the overall stock data
     time_series = parse_stock_data(data) #get the time series data
     time_series_data_frame = convert_to_df(time_series) #convert to the dataframe
     time_series_data_frame = time_serise_analysis_dataset(time_series_data_frame)
     time_series_data_frame = time_serise_feature_engineering(time_series_data_frame)  # create analysis data
     
     stats = compute_all_statistics(time_series_data_frame)  #Extract the stats out of it 

     return {
          "symbol" : symbol ,
          "stats" : stats,
          "Data Sets": time_series_data_frame[:10].to_dict(orient="index") , #return only the first 10 records for now
          
     }




In [41]:
data = get_stock_analysis("AAPL")
symbol = data["symbol"]
final_output = data["Data Sets"]

final_output.keys()



Exception: Time Series data not found in API response